# BIP Trace Collection and Evaluation Basics


The below code is used, to determine the current testline configuration by the host name.

With the host name information, the testline configuration is loaded, so you can later on
work with the common keywords on the testline.

You need to execute the cell (by pressing [Ctrl]+[Enter] on the cell or pressing the play button)

After running the cell, an ip address is printed. If the ip address corresponds to the address of the local machine,
the correct testline configuration is loaded.

In [ ]:
*** Settings ***
Library    jupyter_helper.py
Resource    ../ute_workarounds.robot

*** Test Cases ***
Load TL Config
    Load Local Testline Config
    UTE Cloud Workaround
    Log    HostPC is ${tl.test_pc}
    ${host_ip}=    Get Host IP
    ${local_ip}=   Set Variable   ${tl.test_pc.default_ssh_connection_details.addr}
    Should Be Equal    ${host_ip}   ${local_ip}    The tl config IP ${local_ip} does not fit the host ip ${host_ip}
    
    Set Variable   ${local_ip}
    

After having the testline configuration loaded with the first cell on this page, you can start interacting with your testline. 

The next cell is using the keywords from ute_workarounds to bring the testline in the OnAir status

After executing this Keyword, you are able to proceed with further interactions, like attaching UES, collecting logs, ...

# Test Setup and Teardown to connect to running egate and edaemon

Since Jupyter is instanciating the ute_wtssim again again, with each start of a new cell, the alias from a previous run cell is no more valid. A workaround for this is to define a test setup and a test teardown to setup the utw_wtssim with each test case, and tear it down after each test case.

**Don't run the keywords directly, this will lead to errors.**

In [ ]:
*** Settings ***
Library     ute_wtssim
Resource    ../ute_workarounds.robot
Test Setup      Connect to running egate
Test Teardown   Teardown the wts sim
   

*** Keywords ***
Connect to running egate
    ${wts_alias}=      Set Variable    ap_egate
    ${egate_port}=     Set Variable    ${tl.core.egate_port}
    ${edaemon_alias}=  Set Variable    ap_edaemon
    ${edaemon_port}=   Set Variable    ${tl.core.edaemon_port}
    Set Suite Variable    ${wts_alias}
    Set Suite Variable    ${edaemon_alias}

    ute_wtssim.setup_egate    egate_port=${egate_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${wts_alias}
    ute_wtssim.setup_edaemon    edaemon_port=${edaemon_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${edaemon_alias}
    ${wts_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_info    pass_regexp=Process ID    wts_alias=${wts_alias}
    ${sim_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_state_disp    pass_regexp=The state is ACTIVE    wts_alias=${wts_alias}
    
    Run Keyword If    $wts_status == False or $sim_status == False    Fail   Unable To Connect to running Egate

Teardown the wts sim
    ute_wtssim.teardown_egate     alias=${wts_alias}
    ute_wtssim.teardown_edaemon   alias=${edaemon_alias}


In [ ]:
%%python module JupyterDisplay
import taf.ul.gnb.bip.collection

from IPython.display import display_pretty, Markdown, display, Code, JSON

def jupyter_display_markdown(text):
    display(Markdown(text))
    
def jupyter_display_json(output_object):
    display(JSON(output_object))

def jupyter_display_text(text):
    display(Code(text.decode('utf-8')))

def show(text):
    display(text)

def reload():
    import importlib
    importlib.reload(taf.ul.gnb.bip.collection)

In [ ]:
%sticky library ute_wtssim
%sticky library ute_wtssim.interface
%sticky library ute_wtssim.interfaces
%sticky library ute_wtssim.interfaces.wts
%sticky library ute_common_converter
%sticky library ute_common_converter.to_list
%sticky library ute_common_converter.base
%sticky library ute_common_converter.exception
%sticky library ute_core
%sticky library ute_core.exception
%sticky library ute_common_converter.to_number
%sticky library ute_common_sshforwarder
%sticky library ute_common_sshforwarder.forwarder
%sticky library ute_common_sshforwarder.server
%sticky library ute_common_sshforwarder.util
%sticky library ute_common_sshforwarder.handler
%sticky library ute_common_sshforwarder.exception
%sticky library ute_wtssim.interfaces.common
%sticky library ute_common_store
%sticky library ute_common_store.store
%sticky library ute_common_store.exception
%sticky library ute_common_converter.to_bool
%sticky library ute_common_interface
%sticky library ute_common_interface.scope
%sticky library ute_wtssim.wts
%sticky library ute_wtssim.wts.egate
%sticky library ute_wtssim.wts.common
%sticky library ute_wtssim.exception
%sticky library ute_wtssim.client
%sticky library ute_wtssim.client.telnetclient
%sticky library ute_wtssim.wts.edaemon
%sticky library ute_wtssim.interactor
%sticky library ute_wtssim.interactor.notifier
%sticky library ute_wtssim.interactor.importer
%sticky library ute_wtssim.utils
%sticky library ute_wtssim.utils.settings
%sticky library ute_wtssim.wts.state_machine
%sticky library ute_wtssim.filecollector
%sticky library ute_wtssim.filecollector.filecollector
%sticky library ute_wtssim.filecollector.testpath
%sticky library ute_wtssim.filecollector.exception
%sticky library ute_wtssim.utils.decorator
%sticky library ute_wtssim.luaconfiggenerator
%sticky library ute_wtssim.luaconfiggenerator.luaconfgenerator
%sticky library ute_common_admin
%sticky library ute_common_admin.api
%sticky library ute_common_admin.messages
%sticky library ute_common_admin.logger
%sticky library ute_common_admin.controller
%sticky library ute_common_admin.exception
%sticky library ute_common_admin.util
%sticky library ute_wtssim.wts.iphyvalidator
%sticky library ute_wtssim.interfaces.egate
%sticky library ute_wtssim.interfaces.edaemon
%sticky library taf.alias
%sticky library taf.alias.store
%sticky library taf.alias.store.interface

*** Settings ***
Resource    ../wts_ue.robot
Resource    ../ttitrace.robot


*** Test Cases ***
Test Convert TTI trace from pcap to excelCsv with Ue Action
    Ue Register
    Sleep   5 sec
    Ue Deregister


# Collection of the BIP Trace

Focus of the BIP trace is mainly, to retrieve the trace between L1 and L2. Hence. The location of L1 and L2 and their connection differs on the different HW platforms. Hence, different mirroring mechanisms are needed, to be able to retrieve the bts internal protocol trace.

There exist common keyword for most of bip trace collections.

Additional keywords exist in *taf.ul.gnb.bip.collection*. In future, the mirroring can be triggered by the web element manager. Planned to come with 22R4.

In [ ]:
*** Settings ***
Resource    ../wts_ue.robot

Variables   /home/ute/testline_configuration/7_15_UTE5GTL317
Library     taf.ul.gnb.bip.collection   ${tl}

Resource    ../tcpdump.robot

Library     JupyterDisplay
Library     OperatingSystem

*** Test Cases ***
Test Attach UE and Collect BIP Trace

    ${target_directory}=   Set Variable   ${CURDIR}/BIP_Traces
    Create Directory    ${target_directory}
    
    Use Mirror Script    ${True}
    Start Bip Trace
    Ue Register
    Sleep   5 sec
    Ue Deregister
    ${bip_trace}=   Stop Bip Trace   target_dir=${target_directory}
    
    
    Show   ${bip_trace}
    Set Suite Variable   ${bip_trace}


# Dissector retrieval

The following code is for downloading the dissectors for *MCShark*. 

It is assumed, that the code resides within the location `https://artifactory-hz1.int.net.nokia.com/artifactory/`

The format of the url for the dissector for linux is expected to be:
```
https://artifactory-hz1.int.net.nokia.com/artifactory/
mcshark-local/<bts_version_string>/mcshark-3.4-1.14.1-<bts_version_string>+buster.so
```

The function `build_dissector_url_for_sw_version` builds this string with th given gNodeB software version and returns it.


In [ ]:
%%python module DownloadDissectors

import subprocess
import os
import requests
from IPython.display import display_pretty, Markdown, display, Code, JSON

def get_local_system():
    """
    Calls the command 'uname -a' to check the operating system we are running on
    """
    stdout, _ = subprocess.Popen(['uname', '-a'], stdout=subprocess.PIPE).communicate()
    system_info = stdout.decode('utf-8')
    display(system_info)
    return system_info

def build_dissector_url_for_sw_version(gnb_sw_version):
    gnb_sw_version = str(gnb_sw_version)
    display(gnb_sw_version)
    base_url = 'https://artifactory-hz1.int.net.nokia.com/artifactory/mcshark-local/'
    file_name = 'mcshark-3.4-1.14.1-{}+buster.so'.format(gnb_sw_version)
    display(file_name)
    full_url = base_url + gnb_sw_version + '/' +  file_name
    display(full_url)
    return full_url
    
def download_mcshark_dissector(gnb_sw_version, target_dir='/tmp'):
    url = build_dissector_url_for_sw_version(gnb_sw_version)
    stdout, _ = subprocess.Popen(['wget', url], cwd=target_dir, stdout=subprocess.PIPE).communicate()
    display(stdout)
        


In [ ]:
*** Settings ***
Library    DownloadDissectors
Library    JupyterDisplay
Resource   ../testline.robot

*** Test Cases ***
Check for the download of dissectors
    ${target_directory}=   Set Variable   ${CURDIR}/BIP_Traces
    ${system}=    Get Local System
    ${sw_version}=    Get gNB SW version
    ${url}=   Build Dissector Url for SW Version    ${sw_version}
    Download Mcshark Dissector   ${sw_version}    target_dir=${target_directory}
    



In [ ]:
*** Settings ***
Library     ../python/PcapAnalysis.py
Library     ../python/BipTraceDissectors.py

*** Test Cases ***
Analyse the collected bip trace
    [Arguments]    ${bip_trace}
    ${gnb_sw_version}=    Get gNB SW version
    ${luashark_path}=     download_and_unzip_dissectors   ${gnb_sw_version}
    
    
    ${uldata_pucchreceiverespps_fields}=   Create Dictionary 
    ...             sfn              UlData_PucchReceiveRespPs.PucchReceiveRespPs.sfn
    ...             slot             UlData_PucchReceiveRespPs.PucchReceiveRespPs.slot
    ...             rnti             UlData_PucchReceiveRespPs.pucchResources_item_0.rnti
    ...             startPrb         UlData_PucchReceiveRespPs.pucchResources_item_0.startPrb
    ...             pucchFormat      UlData_PucchReceiveRespPs.pucchResources_item_0.pucchFormat
    
                     
    ${uldata_prachreceivereq_fields}=   Create Dictionary    sfn    UlData_PrachReceiveReq.PrachReceiveReq.sfn
    ...                                                      slot   UlData_PrachReceiveReq.PrachReceiveReq.slot                                      

    ${test_data}=    Create Dictionary   uldata_prachreceivereq       ${uldata_prachreceivereq_fields}
    ...                                  uldata_pucchreceiverespps    ${uldata_pucchreceiverespps_fields}
    
    
    

    ${bip_data}=   Create Analysis Data    pcap_file=${bip_trace}   dissector_path=${luashark_path}   test_data=${test_data}
    
    Should Not be Empty   ${bip_data}    BIP Trace is empty - no BIP Trace detected. PrachReceiveReq should be present.
    ${amount_of_prach_reseive_req}=    Get Length    ${bip_data}
    Should Be True     ${amount_of_prach_reseive_req} > 10    Not enouth Prach Receive Reqs found!

In [ ]:
*** Settings ***
Resource   ../util.robot

*** Test Cases ***
Convert the Bip trace to pdml
    Convert PCAP to PDML   source=${bip_trace}   target=${bip_trace}.pdml
    